# 00_smoke: four go/no-go gates (task S0.5)

Checks that every heavy stage of the pipeline installs and runs on the Colab runtime before any
real work starts. Thin driver: all logic is in the `citecurv` package and the vendored DyGLib.

| Block | Stage | What runs | PASS when |
|---|---|---|---|
| A | curvature (`30_curvature`) | directed Ollivier-Ricci with GraphRicciCurvature 0.6.1, POT 0.9.6, networkit 11.2.1, SciPy <= 1.13.1 | the six toy graphs and the upstream example give the hand-computed values, flipping an edge changes 5 of 6 values, a 2-process pool gives the same values, nothing hangs |
| B | TGN (`40_tgn`) | the **vendored** DyGLib (`third_party/DyGLib`, patches P1-P7 in its `PATCHES.md`) trains TGN for 3 epochs through `citecurv.tgn.train` on a small SYNTHETIC citation stream | training loss is finite and decreases, validation AP > 0.6, the memory leak check finds nothing |
| C | embeddings (`20_embed`) | SPECTER2 on the GPU with exactly `adapters==1.3.0` and `transformers==4.57.6`, through `citecurv.embed.specter2` | output shape (3, 768), and two same-field texts are closer than a cross-field pair |
| D | GNN baselines (`50_eval`) | `pygod==1.1.0` with `torch_geometric==2.6.1` | DOMINANT, CoLA and GADNR import, `gen_structural_outlier(directed=True)` returns the graph and the outlier mask, and a 2-epoch DOMINANT fit gives finite scores |

**Runtime.** GPU (T4) with **runtime version 26.07** (Python 3.12): Runtime > Change runtime type.
Colab's default runtime has been Python 3.13 since 2026-08-25, and the choice of an older version
does not persist between sessions, so check it every time. All four blocks run in one session (not
yet timed on Colab; off Colab, without installs and downloads, blocks A, B and D took about 1 s, 16 s
and 3 s on CPU).

**Before running.** Set `TAG` in the parameters cell to the release tag to test. No Colab Secrets
are needed (SPECTER2 is public on the Hugging Face hub). Run all. If the install cell asks for a
restart, use Runtime > Restart session and Run all again.

**Independence.** Each block records PASS, FAIL or SKIP with its package versions and the error,
then the next block runs anyway. The last cell writes `smoke_report.json` to
`MyDrive/citecurv/`. Copy it into the repository as `results/smoke_report_<date>.json`, and record
the result and the compute units used in `PROGRESS.md` (S0.5 verify line and unit-burn log).
Any FAIL: open the matching risk row in `IMPLEMENTATION_PLAN.md` section 10 before proceeding; see
also `docs/colab_runbook.md` (known failure modes).

Changes against the plan text of S0.5, all from wave-1 findings: Block A uses the 0.6.1 pin
recommended in `docs/orc_version_comparison.md` (PROVISIONAL until Maor decides; 0.5.3.2 gives the
same values on unweighted graphs) and never the ATD method; Block B uses the patched vendored
DyGLib on a synthetic stream instead of a fresh upstream clone with the Wikipedia download; Block C
pins `transformers==4.57.6` exactly; Block D pins PyG 2.6.1 exactly.

In [ ]:
# ---- 1. Python version gate (run this first) ------------------------------------------------------
# Colab's default runtime has been Python 3.13 since 2026-08-25. The curvature pins need 3.12: every
# GraphRicciCurvature release up to 0.6.1 declares scipy<=1.13.1, and SciPy 1.13.1 has no Python 3.13
# wheel. Fix: Runtime > Change runtime type > Runtime version 26.07 (or older; these run Python 3.12),
# keep the T4 GPU, Save, then Run all. The runtime-version choice does not persist between sessions.
#
# ALLOW_PY313 = True is the opt-in fallback (PROVISIONAL, Maor decides): on Python 3.13 Block A
# installs GraphRicciCurvature with --no-deps next to the runtime's SciPy. The package never imports
# SciPy, and this path gave values identical to Python 3.12 off Colab (docs/orc_version_comparison.md,
# section "Python 3.13"). The report records which path ran.
import platform
import sys

ALLOW_PY313 = False

PY = sys.version_info[:2]
print("python  ", sys.version)
print("platform", platform.platform())
if PY != (3, 12) and not (PY == (3, 13) and ALLOW_PY313):
    raise RuntimeError(
        f"This notebook needs Python 3.12, found {platform.python_version()}. In Colab: Runtime > Change "
        "runtime type > Runtime version 26.07 (Python 3.12), Save, then Run all again. To test the "
        "Python 3.13 fallback instead, set ALLOW_PY313 = True in this cell (docs/orc_version_comparison.md).")
print("Python version OK" + (" (Python 3.13 opt-in path)" if PY == (3, 13) else ""))

In [ ]:
# ---- 2. Parameters --------------------------------------------------------------------------------
TAG = "<TAG>"  # release tag (or branch) of the repository to test, e.g. "v0.2.0"
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
DRIVE_DIR = "/content/drive/MyDrive/citecurv"  # smoke_report.json is written here (one small file)

# Pins under test: configs/pins.md plus the 2026-10 refresh (docs/literature_refresh_2026-10.md).
GRC_VERSION = "0.6.1"  # recommended in docs/orc_version_comparison.md; configs/pins.md still lists
                       # 0.5.3.2 (same values on unweighted graphs) until Maor decides - PROVISIONAL
PINS = {
    "A": [f"GraphRicciCurvature=={GRC_VERSION}", "POT==0.9.6", "networkit==11.2.1", "scipy<=1.13.1",
          "networkx"],                                   # Python 3.12
    "A_py313_deps": ["POT==0.9.6", "networkit==11.2.1", "networkx", "python-louvain", "cython", "packaging"],
    "A_py313_nodeps": [f"GraphRicciCurvature=={GRC_VERSION}"],   # installed with --no-deps on 3.13
    "C": ["adapters==1.3.0", "transformers==4.57.6"],    # adapters 1.3.0 requires transformers~=4.57.6
    "D": ["pygod==1.1.0", "torch_geometric==2.6.1"],
}
B_EPOCHS = 3       # Block B: TGN epochs (2 or 3; the loss must have room to decrease)
B_N_WORKS = 2000   # Block B: papers in the SYNTHETIC stream (about 16,000 citation events)
TIME_LIMIT_S = {"A": 300, "B": 1800, "C": 900, "D": 600}  # a block running longer is stopped and FAILs

In [ ]:
# ---- 3. Install citecurv from the tag and each block's pins -------------------------------------------
# One pip call per block, so a failed install affects only that block. torch is NOT installed: blocks
# B-D use the runtime's preinstalled CUDA build (torch 2.x). The repository is also cloned, because the
# vendored DyGLib and configs/ are not part of the installed package. Outside Colab (e.g. the WSL2
# fallback, plan section 9.2) nothing is installed: run the notebook in an environment that has the pins.
import importlib.metadata as md
import os
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def installed(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None


INSTALL = {}


def pip_install(block, *args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    INSTALL[block] = {"cmd": "pip install " + " ".join(args), "ok": r.returncode == 0,
                      "log_tail": "" if r.returncode == 0 else (r.stdout + r.stderr)[-3000:]}
    print(("[ok]     " if r.returncode == 0 else "[FAILED] ") + INSTALL[block]["cmd"])
    if r.returncode:
        print(INSTALL[block]["log_tail"])


if IN_COLAB:
    assert TAG != "<TAG>", "set TAG in the parameters cell first"
    REPO_DIR = Path("/content/citecurv-repo")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", TAG, REPO, str(REPO_DIR)], check=True)
    pip_install("citecurv", f"citecurv @ git+{REPO}@{TAG}")
    if PY == (3, 12):
        pip_install("A", *PINS["A"])
    else:
        pip_install("A", *PINS["A_py313_deps"])
        pip_install("A_nodeps", "--no-deps", *PINS["A_py313_nodeps"])
    pip_install("C", *PINS["C"])
    pip_install("D", *PINS["D"])
    WORK_DIR = Path("/content/smoke_work")
else:
    REPO_DIR = Path(os.environ.get("CITECURV_REPO_DIR", ".")).resolve()
    WORK_DIR = Path(os.environ.get("CITECURV_SMOKE_WORK", "smoke_work")).resolve()
    print("not on Colab: nothing installed; repository at", REPO_DIR)
DYGLIB_ROOT = REPO_DIR / "third_party" / "DyGLib"
os.environ["CITECURV_DYGLIB_ROOT"] = str(DYGLIB_ROOT)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("vendored DyGLib:", DYGLIB_ROOT, "| PATCHES.md present:", (DYGLIB_ROOT / "PATCHES.md").exists())

# A module imported before pip replaced its package keeps the old version until a restart
# (scipy<=1.13.1 needs numpy<2.3, so pip may downgrade numpy; adapters downgrades transformers).
stale = [f"{mod} {sys.modules[mod].__version__} loaded but {installed(mod)} installed"
         for mod in ("numpy", "scipy", "pandas", "networkx", "torch", "transformers")
         if mod in sys.modules and getattr(sys.modules[mod], "__version__", None) not in (None, installed(mod))]
if stale:
    raise RuntimeError("Restart needed: " + "; ".join(stale) + ". Use Runtime > Restart session, then Run all "
                       "again (the installs are done and are skipped quickly).")

In [ ]:
# ---- 4. Drive, report and block runner ----------------------------------------------------------------
import contextlib
import datetime
import json
import signal
import time
import traceback

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")  # mounted now, so no authorisation prompt waits at the end of the run
    OUT_DIR = Path(DRIVE_DIR)
else:
    OUT_DIR = Path(os.environ.get("CITECURV_SMOKE_OUT", str(WORK_DIR)))
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT = {
    "task": "S0.5",
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "tag": TAG,
    "python": platform.python_version(),
    "python_path": "3.12" if PY == (3, 12) else "3.13 opt-in (GraphRicciCurvature --no-deps)",
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG"),
    "pins": PINS,
    "install": INSTALL,
    "blocks": {},
}


@contextlib.contextmanager
def time_limit(seconds):
    """Raise TimeoutError when the body runs longer than `seconds` (e.g. a hung fork pool)."""
    def on_alarm(signum, frame):
        raise TimeoutError(f"block stopped after {seconds} s")

    old = signal.signal(signal.SIGALRM, on_alarm)
    signal.alarm(int(seconds))
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def run_block(name, fn, dists):
    """Run one gate; record PASS / FAIL / SKIP, checks, error and package versions; never raise."""
    print(f"=== Block {name} ===")
    t0, rec = time.time(), {"status": "FAIL"}
    try:
        with time_limit(TIME_LIMIT_S[name]):
            out = fn()
        rec.update(out)
        rec["status"] = "PASS" if out.get("pass") else ("SKIP" if out.get("skipped") else "FAIL")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"
        rec["traceback"] = traceback.format_exc()[-4000:]
    rec["seconds"] = round(time.time() - t0, 1)
    rec["install_ok"] = {k: v["ok"] for k, v in INSTALL.items() if k == "citecurv" or k.startswith(name)}
    rec["versions"] = {d: installed(d) for d in dists}
    REPORT["blocks"][name] = rec
    print(f"Block {name}: {rec['status']}  ({rec['seconds']} s)")
    for k, v in rec.get("checks", {}).items():
        print(f"  check {k:28s} {v}")
    for k, v in rec["versions"].items():
        print(f"  {k:34s} {v}")
    if rec["status"] != "PASS":
        print("  reason:", rec.get("reason") or rec.get("error"))
    return rec

## Block A: directed Ollivier-Ricci curvature (S1.5 values, decision L5)

Toy graphs from `tests/test_forman_directed.py` with the hand-computed directed ORC values of
`tests/test_orc.py` (alpha 0.5: the source measure spreads over the predecessors of u, the target
measure over the successors of v). Expected values: 0, -0.5, -0.5, -0.5, 0.5, -0.5, and 0 for all
six after flipping the tested edge; the upstream directed example gives -0.5, 0, 0.5, 0.5, 0.75.
These match `results/orc_validation_061.json` (measured off Colab with the same library version).

Methods: OTD and OTDSinkhornMix only. **Never ATD**: it divides by |pred(u)| x |succ(v)| and raises
ZeroDivisionError whenever the citing paper is uncited at its own publication time, which is almost
every citation (`docs/orc_version_comparison.md`). The 2-process run checks that the library's
fork pool works on this runtime. Any FAIL here triggers the WSL2 fallback check (plan 9.2).

In [ ]:
def block_a():
    import networkx as nx
    import GraphRicciCurvature
    from citecurv.curvature.orc import orc_directed

    # (edges, tested edge, ORC, ORC after flipping the tested edge), alpha = 0.5
    toys = [
        ([("u", "v")], ("u", "v"), 0.0, 0.0),
        ([("a", "b"), ("b", "c")], ("a", "b"), -0.5, 0.0),
        ([("a", "b"), ("b", "c")], ("b", "c"), -0.5, 0.0),
        ([("x1", "u"), ("x2", "u"), ("x3", "u"), ("u", "v")], ("u", "v"), -0.5, 0.0),
        ([("a", "b"), ("b", "c"), ("c", "a")], ("a", "b"), 0.5, 0.0),
        ([("u", "v"), ("v", "r1"), ("v", "r2"), ("v", "r3"), ("v", "r4"), ("v", "r5")], ("u", "v"), -0.5, 0.0),
    ]
    upstream = {(0, 1): -0.5, (1, 2): 0.0, (1, 3): 0.5, (2, 3): 0.5, (3, 1): 0.75}  # upstream test suite
    tol = 1e-9
    rows = []
    for edges, e, expected, flipped_expected in toys:
        G = nx.DiGraph(edges)
        otd = orc_directed(G, [e], method="OTD")[e]
        mix = orc_directed(G, [e], method="OTDSinkhornMix")[e]
        F = G.copy()
        F.remove_edge(*e)
        F.add_edge(e[1], e[0])
        flipped = orc_directed(F, [(e[1], e[0])], method="OTD")[(e[1], e[0])]
        one = orc_directed(G, list(G.edges), method="OTD", proc=1)
        two = orc_directed(G, list(G.edges), method="OTD", proc=2)
        rows.append({"edges": [f"{a}->{b}" for a, b in edges], "edge": f"{e[0]}->{e[1]}",
                     "expected": expected, "otd": otd, "otd_sinkhorn_mix": mix,
                     "flipped_expected": flipped_expected, "flipped": flipped,
                     "proc2_vs_proc1_max_abs_dev": max(abs(two[k] - one[k]) for k in one)})
    up = orc_directed(nx.DiGraph(list(upstream)), list(upstream), method="OTD")
    checks = {
        "hand_values_otd": all(abs(r["otd"] - r["expected"]) < tol for r in rows),
        "hand_values_mix": all(abs(r["otd_sinkhorn_mix"] - r["expected"]) < tol for r in rows),
        "flipped_values": all(abs(r["flipped"] - r["flipped_expected"]) < tol for r in rows),
        "direction_changes_5_of_6": sum(abs(r["flipped"] - r["otd"]) > tol for r in rows) == 5,
        "upstream_example": all(abs(up[k] - v) < tol for k, v in upstream.items()),
        "proc2_equals_proc1": all(r["proc2_vs_proc1_max_abs_dev"] < tol for r in rows),
    }
    return {"pass": all(checks.values()), "checks": checks, "toys": rows,
            "upstream_values": {f"{a}->{b}": v for (a, b), v in up.items()},
            "graphriccicurvature": GraphRicciCurvature.__version__,
            "methods": "OTD and OTDSinkhornMix; ATD is never used"}


run_block("A", block_a, ["GraphRicciCurvature", "POT", "networkit", "scipy", "numpy", "networkx", "citecurv"])

## Block B: vendored DyGLib TGN under the runtime's torch (S1.7 compatibility gate)

Uses the DyGLib copy vendored in the repository (`third_party/DyGLib`, upstream commit 3aacc36,
patches P1-P7 in its `PATCHES.md`). Do not clone upstream DyGLib: unpatched, it fails on Python >= 3.11
(`random.sample` on a set, P3), on torch >= 2.6 (`torch.load` weights_only, P4), on any feature width
other than 172 (P1) and on dataset names outside its benchmark list (P6).

Data: a SYNTHETIC citation stream from `citecurv.data.synthetic` (2,000 papers, 4 fields, 2016-2020,
64-d stand-in embeddings), converted with `citecurv.tgn.convert.to_jodie`; no Wikipedia download is
needed. Training: `citecurv.tgn.train.train_tgn`, 3 epochs, direction-aware messages (patch P5, as
configured in `tgn.direction_aware`), memory leak check on, learning rate 1e-3, GPU if present.
PASS = training loss finite and lower in the last epoch than in the first, validation AP of the last
epoch > 0.6, zero leak-check violations. All numbers are measurements on synthetic data. For scale:
a CPU run of this block off Colab (torch 2.15) gave training loss 0.540, 0.465, 0.444 and validation
AP 0.789, 0.790, 0.787 on its 16,001 synthetic events.

In [ ]:
def block_b():
    import numpy as np
    import torch
    from citecurv.data.synthetic import generate_citation_stream
    from citecurv.tgn.convert import to_jodie
    from citecurv.tgn.train import train_tgn

    assert B_EPOCHS >= 2, "B_EPOCHS must be 2 or 3: the loss has to be able to decrease"
    stream = generate_citation_stream(n_works=B_N_WORKS, n_fields=4, years=(2016, 2020), refs_mean=8,
                                      emb_dim=64, seed=0)  # SYNTHETIC
    data_dir = WORK_DIR / "tgn_data"
    meta = to_jodie(stream["events"], stream["embeddings"], data_dir, "smokecite")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    m = train_tgn(data_dir, "smokecite", epochs=B_EPOCHS, seed=0, device=device,
                  node_feat_dim=meta["node_feat_dim"], out_dir=WORK_DIR / "tgn_run", dyglib_root=DYGLIB_ROOT,
                  direction_aware=True, leak_check=True, learning_rate=1e-3)
    loss = [e["train_loss"] for e in m["epochs"]]
    val_ap = [e["val_ap"] for e in m["epochs"]]
    leak = m.get("leak_check") or {}
    checks = {
        "epochs_completed": len(loss) == B_EPOCHS,
        "train_loss_finite": bool(np.all(np.isfinite(loss))),
        "train_loss_decreased": len(loss) >= 2 and loss[-1] < loss[0],
        "val_ap_last_gt_0.6": bool(val_ap) and val_ap[-1] > 0.6,
        "leak_check_clean": leak.get("enabled") is True and leak.get("violations") == 0,
    }
    return {"pass": all(checks.values()), "checks": checks, "synthetic": True,
            "dataset": {k: meta[k] for k in ("n_events", "n_nodes", "node_feat_dim", "edge_feat_dim")},
            "device": m.get("device_resolved") or device,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "train_loss": loss, "val_ap": val_ap, "test": m.get("test"), "leak_check": leak,
            "wall_clock_s": m.get("wall_clock_s"), "cuda_peak_mb": m.get("cuda_peak_mb"),
            "dyglib_args": m.get("resolved_args_subset")}


run_block("B", block_b, ["torch", "numpy", "pandas", "scikit-learn", "tqdm", "citecurv"])

## Block C: SPECTER2 on the GPU (S1.6, decision L6)

`citecurv.embed.specter2.load_specter2` loads `allenai/specter2_base` with the `allenai/specter2`
proximity adapter (the second name is the adapter, not a model) and `encode` embeds three short
texts written for this check (they are not real papers): two on learning from dynamic graphs, one on
a clinical study. PASS = the pins are exactly `adapters==1.3.0` and `transformers==4.57.6`, the
output is a finite float32 array of shape (3, 768), and the two same-field texts have a higher cosine
similarity than the cross-field pair. Without a GPU the block reports SKIP (the gate then fails).
The first run downloads about 440 MB from the Hugging Face hub; a warning about a missing `HF_TOKEN`
secret can be ignored (the models are public).

In [ ]:
SPECTER2_TEXTS = [  # (id, title, abstract); written for this check, not real papers
    ("same_1", "Temporal graph networks for link prediction in dynamic citation graphs",
     "We learn node memories from a stream of timestamped edges and predict which papers will cite which."),
    ("same_2", "Anomaly detection on dynamic graphs with temporal graph neural networks",
     "We score unusual edges in evolving networks using node embeddings that change over time."),
    ("cross", "Adjuvant chemotherapy and survival in stage III colon cancer",
     "A retrospective cohort study of overall survival after surgery followed by chemotherapy in patients "
     "with colon cancer."),
]


def block_c():
    import numpy as np
    import torch

    pins = {"adapters": installed("adapters"), "transformers": installed("transformers")}
    pins_exact = pins == {"adapters": "1.3.0", "transformers": "4.57.6"}
    if not torch.cuda.is_available():
        return {"pass": False, "skipped": True, "pins": pins,
                "reason": "no GPU in this runtime: Runtime > Change runtime type > T4 GPU, then Run all"}
    from citecurv.embed.specter2 import encode, load_specter2

    tokenizer, model = load_specter2("cuda")
    vec = encode(SPECTER2_TEXTS, batch_size=8, model=model, tokenizer=tokenizer, device="cuda")
    unit = vec / np.linalg.norm(vec, axis=1, keepdims=True)
    same, cross = float(unit[0] @ unit[1]), float(unit[0] @ unit[2])
    checks = {
        "pins_exact": pins_exact,
        "shape_3x768": tuple(vec.shape) == (3, 768),
        "float32_finite": vec.dtype == np.float32 and bool(np.isfinite(vec).all()),
        "same_field_closer": same > cross,
    }
    return {"pass": all(checks.values()), "checks": checks, "pins": pins,
            "cosine_same_field": round(same, 4), "cosine_cross_field": round(cross, 4),
            "gpu": torch.cuda.get_device_name(0),
            "reason": None if pins_exact else f"embed pins not active: {pins}"}


run_block("C", block_c, ["torch", "transformers", "adapters", "huggingface-hub", "tokenizers", "citecurv"])

## Block D: PyGOD baseline tier and the clique injector (S4.4, S4.5; decisions L17, L18)

Imports the three PyGOD detectors of the GNN baseline tier (DOMINANT, CoLA, GADNR), runs
`pygod.generator.gen_structural_outlier` with `directed=True` on a small random directed graph
(2 cliques of 5 nodes: the mask must mark 10 nodes and 2 x 10 one-directional edges must be added),
and fits DOMINANT for 2 epochs (scores must be finite), so the gate covers training, not only imports.
PyGOD 1.1.0 lists Python only up to 3.11 in its classifiers; this block is where its behaviour on the
runtime's Python and torch is checked.

The exact working combination of PyGOD, PyTorch Geometric and torch for the baseline tier is
recorded in `results/smoke_pygod.json` by the baseline work (S4.4). If this block fails, compare the
versions printed below with that file before changing pins (if the file is not in your checkout yet,
the pins in the parameters cell are the plan's).

In [ ]:
def block_d():
    import torch
    import torch_geometric
    from pygod.detector import DOMINANT, GADNR, CoLA  # noqa: F401 (import check)
    from pygod.generator import gen_structural_outlier
    from torch_geometric.data import Data

    g = torch.Generator().manual_seed(0)
    n_nodes, clique_size, n_cliques = 60, 5, 2
    src = torch.randint(0, n_nodes, (240,), generator=g)
    dst = torch.randint(0, n_nodes, (240,), generator=g)
    keep = src != dst
    data = Data(x=torch.randn(n_nodes, 8, generator=g), edge_index=torch.stack([src[keep], dst[keep]]))
    n_before = int(data.edge_index.shape[1])
    data, outlier = gen_structural_outlier(data, m=clique_size, n=n_cliques, directed=True, seed=1)
    added = int(data.edge_index.shape[1]) - n_before

    detector = DOMINANT(epoch=2, gpu=0 if torch.cuda.is_available() else -1, verbose=0)
    detector.fit(data)
    scores = torch.as_tensor(detector.decision_score_)
    checks = {
        "detectors_import": True,
        "outlier_mask": tuple(outlier.shape) == (n_nodes,) and int(outlier.sum()) == clique_size * n_cliques,
        "directed_clique_edges": added == n_cliques * clique_size * (clique_size - 1) // 2,
        "dominant_fit_finite": tuple(scores.shape) == (n_nodes,) and bool(torch.isfinite(scores).all()),
    }
    return {"pass": all(checks.values()), "checks": checks, "edges_added": added,
            "torch_geometric": torch_geometric.__version__,
            "dominant_device": "cuda" if torch.cuda.is_available() else "cpu"}


run_block("D", block_d, ["pygod", "torch_geometric", "torch", "scipy", "scikit-learn", "networkx", "numpy"])

## Report

Writes `smoke_report.json` (blocks, checks, values, versions, install logs of failed installs) to
`MyDrive/citecurv/`. The S0.5 gate passes only with four PASS. Next steps: copy the file into the
repository as `results/smoke_report_<date>.json`, fill in the S0.5 row and the unit-burn log in
`PROGRESS.md`, and commit.

In [ ]:
# ---- 5. Write smoke_report.json to Drive and print the gate -------------------------------------------
REPORT["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
REPORT["summary"] = {b: REPORT["blocks"].get(b, {}).get("status", "NOT RUN") for b in "ABCD"}
REPORT["gate"] = "PASS" if all(s == "PASS" for s in REPORT["summary"].values()) else "FAIL"
report_path = OUT_DIR / "smoke_report.json"
tmp_path = OUT_DIR / "smoke_report.json.tmp"
tmp_path.write_text(json.dumps(REPORT, indent=2, default=str) + "\n", encoding="utf-8")
os.replace(tmp_path, report_path)
for b, s in REPORT["summary"].items():
    print(f"Block {b}: {s}")
print("S0.5 gate:", REPORT["gate"], "| report:", report_path)